In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
pd.set_option("display.max_info_columns", 150)

from src.database.connection import get_connection

import warnings
warnings.filterwarnings(
    "ignore",
    message = "pandas only supports SQLAlchemy connectable.*",
    category = UserWarning
)

In [ ]:
conn = get_connection()

query = """
    SELECT *
    FROM generation_eda
    ORDER BY start_time
"""

generation = pd.read_sql(query, conn)
generation["publish_time"] = pd.to_datetime(generation["publish_time"], utc = True)
generation["start_time"] = pd.to_datetime(generation["start_time"], utc = True)

query = """
    SELECT *
    FROM weather_eda
"""

weather = pd.read_sql(query, conn)
weather["forecast_time"] = pd.to_datetime(weather["forecast_time"], utc = True)

query = """
    SELECT 
        *
    FROM demand_eda;
"""

demand = pd.read_sql(query, conn)
demand["start_time"] = pd.to_datetime(demand["start_time"], utc = True)

conn.close()

In [ ]:
fill_generation = generation.copy()

fill_generation = generation.sort_values(["fuel_type", "start_time"])
fill_generation["previous_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(1)
fill_generation["next_generation"] = fill_generation.groupby("fuel_type")["generation_mw"].shift(-1)
fill_generation["previous_start_time"] = fill_generation.groupby("fuel_type")["start_time"].shift(1)
fill_generation["interval"] = fill_generation["start_time"] - fill_generation["previous_start_time"]

wrong_intervals = fill_generation[fill_generation["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["publish_time"] = wrong_intervals["start_time"]
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["generation_mw"] = ((
    wrong_intervals["previous_generation"] + wrong_intervals["next_generation"]
) / 2).round(0)

wrong_intervals = wrong_intervals[["publish_time", "start_time", "fuel_type", "generation_mw"]]

generation = pd.concat([generation, wrong_intervals], ignore_index = True).sort_values("start_time")

In [ ]:
fill_demand = demand.copy()

fill_demand["previous_demand"] = fill_demand["true_demand_mw"].shift(1)
fill_demand["next_demand"] = fill_demand["true_demand_mw"].shift(-1)
fill_demand["previous_start_time"] = fill_demand["start_time"].shift(1)
fill_demand["interval"] = fill_demand["start_time"] - fill_demand["previous_start_time"]

wrong_intervals = fill_demand[fill_demand["interval"] == pd.Timedelta(hours = 1)].copy()
wrong_intervals["start_time"] = wrong_intervals["start_time"] - pd.Timedelta(minutes = 30)
wrong_intervals["true_demand_mw"] = ((
    wrong_intervals["previous_demand"] + wrong_intervals["next_demand"]
) / 2).round(0)

wrong_intervals = wrong_intervals[["start_time", "true_demand_mw"]]

demand = pd.concat([demand, wrong_intervals], ignore_index = True).sort_values("start_time")
demand = demand.rename(columns = {"start_time": "prediction_time"})

In [ ]:
demand["demand_lag_30m"] = demand["true_demand_mw"].shift(1)
demand["demand_lag_1h"] = demand["true_demand_mw"].shift(2)
demand["demand_lag_2h"] = demand["true_demand_mw"].shift(4)
demand["demand_lag_6h"] = demand["true_demand_mw"].shift(12)
demand["demand_lag_12h"] = demand["true_demand_mw"].shift(24)

demand["demand_rolling_3h"] = demand["true_demand_mw"].shift(1).rolling(6).mean().round(0)
demand["demand_rolling_6h"] = demand["true_demand_mw"].shift(1).rolling(12).mean().round(0)
demand["demand_rolling_12h"] = demand["true_demand_mw"].shift(1).rolling(24).mean().round(0)

In [ ]:
horizon_dfs = []

for horizon in range(48):
    df = demand.copy()
    df["horizon"] = horizon + 1 
    df["target_time"] = df["prediction_time"] + pd.Timedelta(minutes = 30 * (horizon))

    horizon_dfs.append(df)

modelling = pd.concat(
    objs = horizon_dfs,
    ignore_index = True
)

target_demand = demand[["prediction_time", "true_demand_mw"]].rename(columns = {
    "true_demand_mw": "target_demand"
})

modelling = modelling.merge(
    right = target_demand,
    left_on = "target_time",
    right_on = "prediction_time",
    how = "left"
)

modelling = modelling.drop(columns = [
    "true_demand_mw",
    "prediction_time_y"
]).rename(columns = {
    "prediction_time_x": "reference_time"
})

modelling.head()

In [ ]:
demand_basic = demand[["prediction_time", "true_demand_mw"]].copy()

lag_times = {
    "24h": pd.Timedelta(hours = 24),
    "48h": pd.Timedelta(hours = 48),
    "7d": pd.Timedelta(days = 7)
}

for label, time in lag_times.items():
    lookup = demand_basic.rename(columns = {
        "prediction_time": "lag_time",
        "true_demand_mw": f"demand_lag_{label}"
    })

    modelling["lag_time"] = modelling["target_time"] - time

    modelling = modelling.merge(
        right = lookup,
        on = "lag_time",
        how = "left"
    ).drop(columns = "lag_time")

modelling = modelling.sort_values(["reference_time", "horizon"]).dropna().reset_index(drop = True)

modelling[[
    "reference_time", "target_time", "horizon", "target_demand",
    "demand_lag_24h", "demand_lag_48h", "demand_lag_7d"
]].head()

In [ ]:
generation_pivot = generation.pivot_table(
    index = "publish_time",
    columns = "fuel_type",
    values = "generation_mw",
    aggfunc = "last"
).reset_index().sort_values("publish_time").drop(columns = "OIL")

generation_pivot.head()

In [ ]:
fuels = [
    "BIOMASS",
    "WIND",
    "PS",
    "OTHER",
    "OCGT",
    "NPSHYD",
    "NUCLEAR",
    "COAL",
    "CCGT"
]

interconnectors = [
    "INTNSL",
    "INTNEM",
    "INTIRL",
    "INTIFA2",
    "INTFR",
    "INTEW",
    "INTELEC",
    "INTNED"
]

generation_pivot["total_fuel"] = generation_pivot[fuels].sum(axis = 1)
generation_pivot["total_interconnector"] = generation_pivot[interconnectors].sum(axis = 1)
generation_pivot["total_generation"] = generation_pivot["total_fuel"] + generation_pivot["total_interconnector"]

features_for_lag = fuels + interconnectors + ["total_fuel", "total_interconnector", "total_generation"]

for feature in features_for_lag:
    generation_pivot[f"{feature}_lag_30m"] = generation_pivot[feature].shift(1)
    generation_pivot[f"{feature}_lag_1h"] = generation_pivot[feature].shift(2)
    generation_pivot[f"{feature}_lag_2h"] = generation_pivot[feature].shift(4)

generation_pivot = generation_pivot.rename(columns = {
    "publish_time": "reference_time"
})

generation_pivot[[
    "total_fuel", "total_interconnector", "total_generation",
    "BIOMASS_lag_30m", "BIOMASS_lag_1h", "BIOMASS_lag_2h"
]].head()

In [ ]:
modelling = pd.merge_asof(
    left = modelling,
    right = generation_pivot,
    on = "reference_time",
    direction = "backward"
)

modelling.head()

In [ ]:
weather = weather.drop(columns = "apparent_temperature")

weather_pivot = weather.pivot(
    index = "forecast_time",
    columns = "location_name",
    values = [
        "temperature_2m",
        "relative_humidity_2m",
        "snowfall",
        "rain"
    ]
).reset_index()

weather_pivot.columns = [
    f"{weather_condition}_{city}".lower().replace(" ", "_")
    for weather_condition, city in weather_pivot.columns
]

weather_pivot.head()

In [ ]:
temperature_columns = [
    column for column in weather_pivot.columns
    if column.startswith("temp")
]

rain_columns = [
    col for col in weather_pivot.columns
    if col.startswith("rain")
]

snow_columns = [
    col for col in weather_pivot.columns
    if col.startswith("snow")
]

weather_pivot["temperature_mean"] = weather_pivot[temperature_columns].mean(axis = 1)
weather_pivot["temperature_min"] = weather_pivot[temperature_columns].min(axis = 1)
weather_pivot["temperature_max"] = weather_pivot[temperature_columns].max(axis = 1)
weather_pivot = weather_pivot.drop(columns = temperature_columns)

weather_pivot["cities_with_rain"] = (weather_pivot[rain_columns] > 0).sum(axis = 1)
weather_pivot["cities_with_snow"] = (weather_pivot[snow_columns] > 0).sum(axis = 1)

weather_pivot = weather_pivot.rename(columns = {
    "forecast_time_": "target_time"
})

weather_pivot[[
    "target_time", 
    "temperature_mean", "temperature_min", "temperature_max",
    "cities_with_rain", "cities_with_snow"
]].head()

In [ ]:
modelling = modelling.sort_values("target_time")

modelling = pd.merge_asof(
    left = modelling,
    right = weather_pivot,
    on = "target_time",
    direction = "backward"
)

modelling = modelling.sort_values(["reference_time", "horizon"]).reset_index(drop = True)

modelling.head()

In [ ]:
def months_to_seasons(month):
    if 3 <= month < 6:
        return "Spring"
    elif 6 <= month < 9:
        return "Summer"
    elif 9 <= month < 12:
        return "Autumn"
    else:
        return "Winter"

modelling["month"] = modelling["target_time"].dt.month
modelling["season"] = modelling["target_time"].dt.month.apply(months_to_seasons)

modelling["hour"] = modelling["target_time"].dt.hour
modelling["minute"] = modelling["target_time"].dt.minute
modelling["time_of_day"] = modelling["hour"] + modelling["minute"] / 60
modelling = modelling.drop(columns = [
    "hour",
    "minute"
])

modelling["is_weekday"] = (modelling["target_time"].dt.dayofweek >= 5)

modelling[[
    "reference_time", "target_time", "horizon", "target_demand", 
    "month", "season", "time_of_day", "is_weekday"
]].head()

In [ ]:
modelling.info()